<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [6]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [8]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

**Weak — “add a search feature”**

The assistant invented an architecture: a SearchService, a new index package, embeddings, and tests across several files. It did not read AGENTS.md. It did not name `search_documents`. It started editing before a plan.

**Project-aware — plan only, after AGENTS.md**

Plan (no edits in that turn): one file, `src/bootcamp_agent/tools.py`. Add optional `tags: Sequence[str] | None = None` to `search_documents`. Retrieve first, then filter by any overlapping document tag, leave `MAX_SEARCH_RESULTS` as the retrieve cap. No new dependency. No `.github/`. No `tests/` (not in this copy).

**Difference:** the weak prompt owned the design. The project-aware prompt waited, named the file, and left the decision with me.

## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [x] Ask for a **plan** first. Read it. Restrict files it may touch.
- [x] Ask for the **smallest implementation**.
- [x] Inspect the **diff** yourself, line by line.
- [x] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [x] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [x] Ask for a summary of remaining risks.

**Rejected change + reason:** The assistant also offered to update `.github/workflows/pages.yml` so the site rebuilds on push. Rejected: that file is outside the allowed set (`src/bootcamp_agent/tools.py` only). It is unpublished for students and publishes the course site — out of scope, not a missing-file coincidence. The assistant then refused the edit. Also rejected: adding a pytest file; `tests/` is not in this copy.

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

**What was assumed:** the tags filter used OR (any overlap) without asking if it should be AND. Also: a helpful extra file under `.github/`.

**Sentence added to AGENTS.md (Before editing):** if a list filter is underspecified (any vs all), ask; do not pick a rule in silence. Do not edit `.github/` unless those files were named in the allowed set.

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [9]:
loop = {
    "plan_approved": (
        "Approved a one-file plan: only src/bootcamp_agent/tools.py. "
        "Add optional tags to search_documents, filter after retrieve, "
        "leave MAX_SEARCH_RESULTS as the retrieve cap, no new dependency."
    ),
    "diff_inspected": (
        "Read the search_documents signature: added tags: Sequence[str] | None = None. "
        "After retrieve, if tags is non-empty, keep scored chunks whose document tags "
        "intersect the wanted set (OR). Empty tags skips the filter. Return string unchanged. "
        "Tool description now mentions the optional filter. No other files in the diff."
    ),
    "rejected_change": (
        "Refused editing .github/workflows/pages.yml to rebuild the site on push, "
        "and refused adding a pytest file under tests/."
    ),
    "why_rejected": (
        "Out of scope: allowed set was tools.py only. .github is unpublished student-side "
        "and publishes the course site. tests/ is withheld because it holds solved exercises."
    ),
    "risks": (
        "OR vs AND was assumed, not specified — a query with two tags may return extra hits. "
        "Unknown tags silently yield no matches instead of ToolError. Filter runs after retrieve, "
        "so a tagged doc that missed top_k never appears. Cap still applies to retrieve, not to post-filter count."
    ),
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      Approved a one-file plan: only src/bootcamp_agent/tools.py
diff_inspected     Read the search_documents signature: added tags: Sequence[
rejected_change    Refused editing .github/workflows/pages.yml to rebuild the
why_rejected       Out of scope: allowed set was tools.py only. .github is un
risks              OR vs AND was assumed, not specified — a query with two ta


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [10]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [11]:
    review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.